# Partial states: silence, transitions, and the beam as a process

One run of utter's `scripts/partial_states.py`: built streams of Speech Commands words with chosen pauses and finishes cut from the background recordings, decoded at 40 ms blocks with eight readings and partial words on. Tables: `streams`, `words` (each word's position and energy onset/offset in the stream), `gaps` (pause or finish, where, how long, from which recording), `finals` (where the decoder ended an utterance), `stable` (rank 0's hold per block), and `advances` (one row per decoder advance and reading: lead, velocity, age, relation to rank 0, and the trailing `[sil]` span on the best path).

The first figure is the one the tables exist for: a whole stream as a process, every reading's standing over time against what was actually said.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

RUN = Path("../data/2026-09-12-states")
manifest = json.loads((RUN / "MANIFEST.json").read_text())
T = {name: pd.read_parquet(RUN / f"{name}.parquet") for name in ("streams", "words", "gaps", "finals", "stable", "advances")}
adv, words, gaps, finals, stable = (T[k] for k in ("advances", "words", "gaps", "finals", "stable"))
test = lambda df: df[df.split == "testing"]
MS = 16
REL = {"same": "#2a78d6", "extends": "#eb6834", "differs": "#1baf7a", "prefix": "#4a3aa7"}   # categorical slots 1, 2, 3, 7
print(json.dumps({k: manifest[k] for k in ("exported", "harness", "harness_args", "wheel", "utter_head", "utter_dirty")}, indent=1))
T["streams"].groupby("split").agg(streams=("key", "size"), minutes=("samples", lambda s: round(s.sum() / 16000 / 60, 1)), words=("n_words", "sum"), finals=("n_finals", "sum"), advances=("n_advances", "sum"))

## One stream as a process

Top: every reading that ever stood in the top three, its lead over the best other reading at each advance, coloured by its relation to rank 0 at that advance. Word spans (energy onset to offset) are shaded; pauses and finishes are the gaps between them; dashed verticals are the decoder's finals. Bottom: the trailing `[sil]` span on the best path, the end-of-speech clock, and rank 0's hold. Zoom into any transition.

In [ ]:
def stream_view(key, start_ms=None, end_ms=None):
    a = adv[adv.key == key]
    if start_ms is not None: a = a[a.ms >= start_ms]
    if end_ms is not None: a = a[a.ms <= end_ms]
    w = words[words.key == key]; g = gaps[gaps.key == key]; f = finals[finals.key == key]; st = stable[stable.key == key]
    shown = set(a[a["rank"] <= 2].text)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.72, 0.28], vertical_spacing=0.04)
    for _, ww in w.iterrows():
        fig.add_vrect(x0=ww.onset / MS, x1=ww.offset / MS, fillcolor="#2a78d6", opacity=0.08, line_width=0, row=1, col=1,
                      annotation_text=ww.label, annotation_position="top left", annotation_font_size=10)
    for _, ff in f.iterrows():
        fig.add_vline(x=ff.fed / MS, line=dict(color="#898781", dash="dash", width=1))
    for text, s in a[a.text.isin(shown)].groupby("text", sort=False):
        s = s.sort_values("adv")
        fig.add_trace(go.Scatter(x=s.ms, y=s.lead, mode="lines+markers", name=text, line=dict(width=1.5),
                                 marker=dict(size=6, color=[REL[r] for r in s.relation]),
                                 customdata=np.stack([s.lead_delta.round(2), s.relation, s.age_ms], axis=1),
                                 hovertemplate=f"{text}<br>%{{x:.0f}} ms<br>lead %{{y:.2f}}<br>velocity %{{customdata[0]}}<br>%{{customdata[1]}}, age %{{customdata[2]}} ms<extra></extra>"),
                      row=1, col=1)
    fig.add_hline(y=0, line=dict(color="#c3c2b7", width=1), row=1, col=1)
    top = a[a["rank"] == 0].sort_values("adv")
    fig.add_trace(go.Scatter(x=top.ms, y=top.sil_span_ms, mode="lines", name="trailing [sil] span (ms)", line=dict(color="#898781", width=2)), row=2, col=1)
    if len(st):
        stt = st.sort_values("fed"); sel = stt if start_ms is None else stt[(stt.fed / MS >= (start_ms or 0)) & (stt.fed / MS <= (end_ms or 1e12))]
        fig.add_trace(go.Scatter(x=sel.fed / MS, y=sel.stable_ms, mode="lines", name="rank 0 stable_ms", line=dict(color="#eb6834", width=1.5)), row=2, col=1)
    fig.update_layout(template="plotly_white", height=680, title=f"{key}: marker colour = relation to rank 0 (blue same, orange extends, green differs, violet prefix)",
                      legend=dict(orientation="h", y=-0.08), hovermode="closest")
    fig.update_yaxes(title_text="lead over the best other reading (nats)", row=1, col=1)
    fig.update_yaxes(title_text="ms", row=2, col=1); fig.update_xaxes(title_text="ms of audio fed", row=2, col=1)
    return fig

stream_view("testing/0", 0, 25000)

## Kept silence: the hover

Inside finish gaps, well clear of the word before and the word after (480 ms either side), and on the recordings alone: what rank 0 is, how far it leads, and how fast that lead moves. The hypothesis was that silence is kept as a hovering lead, not a growing one.

In [ ]:
a = test(adv); w = test(words); g = test(gaps)
top = a[a["rank"] == 0].copy()
# a finish-gap interior: after the gap's word offset + 480 ms and before the next onset - 480 ms
interior = []
for key, gg in g[g.kind == "finish"].groupby("key"):
    ww = w[w.key == key].sort_values("pos")
    for _, row in gg.iterrows():
        prev = ww[ww.utt <= row.after].tail(1); nxt = ww[ww.pos > row.pos].head(1)
        lo = (prev.offset.iloc[0] if len(prev) else row.pos) / MS + 480
        hi = (nxt.onset.iloc[0] / MS - 480) if len(nxt) else np.inf
        sel = top[(top.key == key) & (top.ms >= lo) & (top.ms <= hi)]
        interior.append(sel)
interior = pd.concat(interior)
print(f"finish-gap interiors: {len(interior)} advances; rank 0 is [sil] on {(interior.text == '[sil]').mean():.1%}")
sil = interior[interior.text == "[sil]"]
print(f"[sil] lead: mean {sil.lead.mean():.2f} sd {sil.lead.std():.2f}; velocity mean {sil.lead_delta.mean():+.3f}, p50 {sil.lead_delta.median():+.3f}")
fig = make_subplots(rows=1, cols=2, subplot_titles=("[sil] lead in kept silence (nats)", "[sil] velocity in kept silence (nats per advance)"))
fig.add_trace(go.Histogram(x=sil.lead, nbinsx=60, marker_color="#2a78d6", name="lead"), row=1, col=1)
fig.add_trace(go.Histogram(x=sil.lead_delta.dropna(), nbinsx=80, marker_color="#eb6834", name="velocity"), row=1, col=2)
fig.update_layout(template="plotly_white", height=380, showlegend=False, bargap=0.05)
fig

## Transition-aligned velocity

Every word onset in the testing streams aligned at advance 0 (the first advance after the energy onset): the mean velocity of the `[sil]` reading while it leads, of whatever leads, and of the best `extends` reading, from three advances before to two after. The question is whether anything moves *before* the onset.

In [ ]:
rows = []
a_sorted = a.sort_values(["key", "adv"])
for key, ww in w.groupby("key"):
    ak = a_sorted[a_sorted.key == key]
    advs = ak.drop_duplicates("adv")[["adv", "fed"]].reset_index(drop=True)
    for _, word in ww.iterrows():
        k0 = advs.index[advs.fed >= word.onset]
        if not len(k0): continue
        k0 = int(k0[0])
        for off in range(-3, 3):
            k = k0 + off
            if k < 0 or k >= len(advs): continue
            at = ak[ak.adv == advs.adv[k]]
            lead0 = at[at["rank"] == 0]
            silr = at[(at.text == "[sil]") & (at["rank"] == 0)]
            ext = at[at.relation == "extends"].sort_values("rank").head(1)
            rows.append(dict(offset=off, series="[sil] while leading", v=silr.lead_delta.iloc[0] if len(silr) else np.nan))
            rows.append(dict(offset=off, series="rank 0", v=lead0.lead_delta.iloc[0] if len(lead0) else np.nan))
            rows.append(dict(offset=off, series="best extends", v=ext.lead_delta.iloc[0] if len(ext) else np.nan))
al = pd.DataFrame(rows)
summ = al.groupby(["series", "offset"]).v.agg(mean="mean", q25=lambda s: s.quantile(.25), q75=lambda s: s.quantile(.75), n="count").reset_index()
print(summ.pivot(index="offset", columns="series", values="mean").round(2))
fig = go.Figure()
cols = {"[sil] while leading": "#4a3aa7", "rank 0": "#2a78d6", "best extends": "#eb6834"}
for name, s in summ.groupby("series"):
    fig.add_trace(go.Scatter(x=s.offset, y=s.q75, mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=s.offset, y=s.q25, mode="lines", line=dict(width=0), fill="tonexty", fillcolor=cols[name] + "22", showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=s.offset, y=s["mean"], mode="lines+markers", name=name, line=dict(color=cols[name], width=2.5), marker=dict(size=9)))
fig.add_vline(x=0, line=dict(color="#898781", dash="dash")); fig.add_hline(y=0, line=dict(color="#c3c2b7", width=1))
fig.update_layout(template="plotly_white", height=460, xaxis=dict(title="advances from the word's energy onset", tickvals=list(range(-3, 3))),
                  yaxis_title="velocity, mean and interquartile band (nats per advance)")
fig

## Momentum by relation

Consecutive velocities of the same reading, 1-4 nats from the lead, split by what the reading is to rank 0. The empty reading's lead persists; a rival's reverts.

In [ ]:
s = a.sort_values(["key", "text", "adv"]).copy()
grp = s.groupby(["key", "text"])
s["next_delta"] = grp.lead_delta.shift(-1); s["next_adv"] = grp.adv.shift(-1)
pairs = s[(s.next_adv == s.adv + 1) & s.lead_delta.notna() & s.next_delta.notna() & s.lead.notna()].copy()
pairs["band"] = pd.cut(pairs.lead.abs(), [0, 1, 4, np.inf], labels=["<1", "1-4", ">=4"], right=False)
band = pairs[pairs.band == "1-4"]
print(f"1-4 nat band, all readings: n={len(band)} r={band.lead_delta.corr(band.next_delta):+.3f}")
for rel, gg in band.groupby("relation"):
    print(f"  {rel:8s}: n={len(gg):6d} r={gg.lead_delta.corr(gg.next_delta):+.3f}")
fig = go.Figure()
for rel in ("same", "extends", "differs", "prefix"):
    gg = band[band.relation == rel]
    fig.add_trace(go.Scattergl(x=gg.lead_delta, y=gg.next_delta, mode="markers", name=f"{rel} (n={len(gg)})",
                               marker=dict(color=REL[rel], size=5, opacity=0.45), text=gg.key + "  " + gg["text"],
                               hovertemplate="%{text}<br>now %{x:.2f}<br>next %{y:.2f}<extra></extra>"))
lim = float(np.nanpercentile(np.abs(band[["lead_delta", "next_delta"]].values), 99.5))
fig.add_shape(type="line", x0=-lim, y0=-lim, x1=lim, y1=lim, line=dict(color="#c3c2b7", dash="dot"))
fig.update_layout(template="plotly_white", height=560, xaxis=dict(title="velocity at advance k (nats)", range=[-lim, lim]),
                  yaxis=dict(title="velocity at advance k+1 (nats)", range=[-lim, lim]))
fig

## The end-of-speech trade

For every silent stretch after a word, the trailing `[sil]` span at each advance against whether the stretch was a pause (another word came) or a finish. A bound on the span calls a finish; the curve is what each bound mistakes and catches. The built pause lengths are uniform 100-800 ms, which drives these numbers; TD-8's private figure comes from real pauses.

In [ ]:
rows = []
for key, gg in g.groupby("key"):
    ww = w[w.key == key].sort_values("pos"); tk = top[top.key == key].sort_values("adv")
    for _, row in gg.iterrows():
        prev = ww[ww.utt <= row.after].tail(1)
        if not len(prev): continue
        start = prev.offset.iloc[0]; end = row.pos + row.samples
        sel = tk[(tk.fed >= start) & (tk.fed <= end)]
        rows.append(dict(kind=row.kind, key=key, gap_ms=row.samples / MS, max_span=sel.sil_span_ms.max() if len(sel) else 0))
sp = pd.DataFrame(rows)
bounds = np.arange(100, 1300, 50)
curve = pd.DataFrame({"bound_ms": bounds,
                      "pauses mistaken": [(sp[sp.kind == "pause"].max_span >= b).mean() for b in bounds],
                      "finishes called": [(sp[sp.kind == "finish"].max_span >= b).mean() for b in bounds]})
fig = go.Figure()
fig.add_trace(go.Scatter(x=curve.bound_ms, y=curve["finishes called"], name="finishes called", mode="lines+markers", line=dict(color="#2a78d6", width=2.5)))
fig.add_trace(go.Scatter(x=curve.bound_ms, y=curve["pauses mistaken"], name="pauses mistaken for a finish", mode="lines+markers", line=dict(color="#eb6834", width=2.5)))
for b, lab in ((300, "TD-8's 300 ms"), (500, "fitted 500 ms")):
    fig.add_vline(x=b, line=dict(color="#898781", dash="dash"), annotation_text=lab, annotation_position="top")
fig.update_layout(template="plotly_white", height=440, xaxis_title="bound on the trailing [sil] span (ms)", yaxis=dict(tickformat=".0%", title="share"))
print(f"pauses {int((sp.kind=='pause').sum())}, finishes {int((sp.kind=='finish').sum())}; at 300 ms: {curve.loc[curve.bound_ms==300, 'pauses mistaken'].iloc[0]:.1%} mistaken, {curve.loc[curve.bound_ms==300, 'finishes called'].iloc[0]:.1%} called")
fig